In [ ]:
# Phase 2 prep: turn raw DICOM studies into one compact .npz artifact each
# (up to MAX_SERIES series x K_SLICES slices, SIZE x SIZE, JPEG q=92), so the
# ~42 experiments Phases 4-5 budget stop paying a DICOM decode per study per
# epoch. CPU-only, internet off -- this is a decode/resize pass, not training,
# so it doesn't touch the GPU quota Phase 3's LLM run needs.
#
# Run this first with PILOT_N set (the 50-study pilot). Four gates have to pass
# on the pilot before the full sharded run generates ~5GB: loader latency,
# the notebook-to-notebook artifact hop (separate consumer kernel), decode
# coverage, and the size budget. A failed gate re-runs the pilot; it does not
# proceed with a caveat.
import glob, os, shutil, sys, time

GIT_SHA = '9800bdc-wip'

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
SRC = src_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import StudyDecodeError
from knee.prep import prep_study, save_study_npz
print('knee package imported successfully from', PKG, '| GIT_SHA', GIT_SHA)

In [ ]:
# Pilot vs. full run is one constant. Sharding is by contiguous index range over
# sorted study UIDs, and it is the resume mechanism, not a fallback for one:
# /kaggle/working starts empty on every run, so "skip studies already prepped"
# only helps within a single session and cannot survive the 12h cap. A lost
# shard re-runs alone.
PILOT_N = None        # 50 for the pilot; None for the full run
SHARD_INDEX = 3       # per-shard kernels differ only in this line
N_SHARDS = 4          # ~1,102 studies / ~35 min each at the pilot's 1.9 s/study

K_SLICES = 24
SIZE = 256
MAX_SERIES = 4

OUT_DIR = '/kaggle/working/prepped'
os.makedirs(OUT_DIR, exist_ok=True)

import math
import numpy as np
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
series_df = pd.read_csv(f'{COMP_DIR}/train_series.csv')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_uids = set(train_df.loc[train_df[gold_cols].notna().any(axis=1), 'StudyInstanceUID'])
print(f'{len(train_df)} total studies, {len(gold_uids)} gold-labeled')

TRAIN_SERIES_DIR = f'{COMP_DIR}/train_series'
all_study_uids = sorted(
    d for d in os.listdir(TRAIN_SERIES_DIR)
    if os.path.isdir(os.path.join(TRAIN_SERIES_DIR, d))
)

per_shard = math.ceil(len(all_study_uids) / N_SHARDS)
lo = SHARD_INDEX * per_shard
study_uids = all_study_uids[lo:lo + per_shard]

if PILOT_N is not None:
    # Spread the pilot across the corpus rather than taking a contiguous prefix:
    # the gates below draw corpus-wide conclusions (one transfer syntax, no
    # MONOCHROME1, no decode failures) and one contiguous block is a weak basis
    # for that -- the laterality census covered all 4,407 studies. Union in the
    # gold studies as well: they are the only ones carrying real labels, and a
    # prefix sample happened to catch just 1 of the 58.
    stride = max(len(study_uids) // PILOT_N, 1)
    sampled = study_uids[::stride][:PILOT_N]
    study_uids = sorted(set(sampled) | {u for u in study_uids if u in gold_uids})

print(f'{len(all_study_uids)} studies on disk')
print(f'shard {SHARD_INDEX}/{N_SHARDS}: index range [{lo}, {lo + per_shard}) '
      f'-> {len(study_uids)} studies this run '
      f'({sum(u in gold_uids for u in study_uids)} gold)')

In [ ]:
# The prep loop. Diagnostic counters stay on for the full run too -- they cost
# nothing and turn the corpus-wide numbers into a free byproduct.
meta_rows = []
failed_studies = []
t0 = time.time()

for i, study_uid in enumerate(study_uids):
    try:
        series_slices, meta = prep_study(
            study_uid,
            TRAIN_SERIES_DIR,
            series_df,
            k_slices=K_SLICES,
            size=SIZE,
            max_series=MAX_SERIES,
            is_gold=study_uid in gold_uids,
        )
        save_study_npz(os.path.join(OUT_DIR, f'{study_uid}.npz'), series_slices, meta)
        meta_rows.append(meta)
    except Exception as exc:
        # Broad on purpose. StudyDecodeError is the expected case (no usable
        # series), but select_k_evenly_spaced raises AssertionError by design, a
        # malformed series_df row raises KeyError, and a 320-slice series can
        # raise MemoryError -- none of which should abort a multi-hour shard at
        # study 4,000 and lose every artifact with it. Each failure is recorded
        # in prep_failed_shard*.csv, so a study is visible as failed rather than
        # silently missing from the artifact count.
        failed_studies.append({
            'StudyInstanceUID': study_uid,
            'error': f'{type(exc).__name__}: {exc}',
        })
    if (i + 1) % 25 == 0:
        elapsed = time.time() - t0
        print(f'{i + 1}/{len(study_uids)} prepped, {elapsed:.1f}s elapsed '
              f'({elapsed / (i + 1) * 1000:.0f} ms/study)')

elapsed = time.time() - t0
print(f'done: {len(meta_rows)} prepped, {len(failed_studies)} failed, in {elapsed:.1f}s')
for row in failed_studies[:10]:
    print(row)
assert meta_rows, 'no study prepped -- every gate below would report on an empty set'

In [ ]:
# GATE 4 -- size budget. The plan estimates 0.7-1.5 MB/study; anything well
# outside that means K or SIZE is wrong and the pilot re-runs rather than the
# full corpus inheriting the mistake.
sizes_mb = np.array([
    os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    for f in os.listdir(OUT_DIR) if f.endswith('.npz')
])
ms_per_study = elapsed / max(len(study_uids), 1) * 1000
n_all = len(all_study_uids)

print(f'artifacts written: {len(sizes_mb)} (expected {len(meta_rows)})')
print(f'MB/study: mean {sizes_mb.mean():.2f}, p50 {np.percentile(sizes_mb, 50):.2f}, '
      f'p95 {np.percentile(sizes_mb, 95):.2f}, max {sizes_mb.max():.2f}')
print(f'extrapolated total for {n_all} studies: {sizes_mb.mean() * n_all / 1000:.1f} GB')
print(f'{ms_per_study:.0f} ms/study -> {ms_per_study * n_all / 3.6e6:.1f} h for the full corpus')
print(f'at a 12h cap with 2x margin, use N_SHARDS >= {math.ceil(ms_per_study * n_all / 3.6e6 / 6)}')

In [ ]:
# GATE 3 -- decode coverage. Phase 1 only ever decoded one series per study
# (sagittal fluid-sensitive). Four series per study walks into transfer
# syntaxes and conventions it never touched, and the failure mode this catches
# is 4,407 artifacts with a third of their series silently empty. Every
# distinct value below has to be accounted for before the full run.
series_meta_rows = []
for meta in meta_rows:
    for series_uid, sm in meta['series'].items():
        series_meta_rows.append({'StudyInstanceUID': meta['StudyInstanceUID'],
                                 'SeriesInstanceUID': series_uid, **sm})
series_meta_df = pd.DataFrame(series_meta_rows)

failures = [f for meta in meta_rows for f in meta['decode_failures']]
skipped = [(meta['StudyInstanceUID'], s, reason)
           for meta in meta_rows for s, reason in meta['skipped_series'].items()]

print(f'series stored: {len(series_meta_df)} across {len(meta_rows)} studies')
print()
print('TransferSyntaxUID:')
print(series_meta_df['TransferSyntaxUID'].value_counts(dropna=False))
print()
print('PhotometricInterpretation (MONOCHROME1 is inverted and percentile_clip_to_uint8')
print('does not correct for it -- any count > 0 here is a code fix, not a note):')
print(series_meta_df['PhotometricInterpretation'].value_counts(dropna=False))
print()
print('matrix size (Rows, Columns) -- non-square means a naive resize to a square')
print('distorts the image, differently per study, so it decides pad-to-square:')
print(series_meta_df.groupby(['Rows', 'Columns']).size().sort_values(ascending=False).head(15))
print()
print('PixelSpacing distinct values:', series_meta_df['PixelSpacing'].astype(str).nunique())
print(series_meta_df['PixelSpacing'].astype(str).value_counts().head(10))
print()
print('series stored per study:')
print(pd.Series([len(m['series']) for m in meta_rows]).value_counts().sort_index())
print()
print(f'decode/header failures: {len(failures)}')
if failures:
    print(pd.DataFrame(failures).groupby(['stage', 'TransferSyntaxUID', 'error'],
                                         dropna=False).size())
print(f'series skipped entirely: {len(skipped)}')
for row in skipped[:10]:
    print(row)
print()
print('laterality route over this shard (should match results/laterality_census.csv):')
print(pd.Series([m['route'] for m in meta_rows]).value_counts())

In [ ]:
# GATE 1 -- loader latency, measured here, not after the full run. Measured
# across the configurations that actually get used, because the cost is roughly
# linear in blobs decoded and the two configurations differ 3x: full training
# (MAX_SERIES x K_SLICES = 96 slices) and the efficiency-track submission
# (2 series x 16). load_study_npz decodes only what the caller asks for, so the
# small configuration does not pay for slices it discards.
#
# The <20ms/study target came from the plan before the cost breakdown was known.
# If the full training configuration misses it, the fix is NOT to cut K -- that
# trades training signal for a loader problem. Report the number and decide.
from knee.dataset import PreppedStudyDataset

# timing over at most 100 studies: this is a latency measurement, and on a full
# shard the three configurations would otherwise add minutes for no extra
# information
prepped_uids = [m['StudyInstanceUID'] for m in meta_rows][:100]

for n_series, n_slices in [(MAX_SERIES, K_SLICES), (2, 16), (2, 12)]:
    loader = PreppedStudyDataset(prepped_uids, OUT_DIR, n_slices=n_slices, max_series=n_series)
    per_study_ms = []
    for idx in range(len(loader)):
        t = time.time()
        image, _, _ = loader[idx]
        per_study_ms.append((time.time() - t) * 1000)
    per_study_ms = np.array(per_study_ms)
    label = 'training' if (n_series, n_slices) == (MAX_SERIES, K_SLICES) else 'efficiency'
    print(f'{n_series} series x {n_slices} slices ({label}): volume {tuple(image.shape)}, '
          f'mean {per_study_ms.mean():.1f} ms/study, p50 {np.percentile(per_study_ms, 50):.1f}, '
          f'p95 {np.percentile(per_study_ms, 95):.1f} '
          f'-- {"PASS" if per_study_ms.mean() < 20 else "over 20ms"}')

# For comparison: what the raw-DICOM path this replaces costs per study.
print(f'\nprep (raw DICOM decode) cost the same studies {ms_per_study:.0f} ms/study')

In [ ]:
# Manifest + the gold holdout list. is_gold lives in every meta, but the Phase 1
# gate-closure note makes the gold holdout a hard requirement before any Phase
# 3/4 pretraining, and a flat CSV makes it greppable without opening 4,407
# archives.
manifest = pd.DataFrame([
    {
        'StudyInstanceUID': m['StudyInstanceUID'],
        'side': m['side'],
        'route': m['route'],
        'is_gold': m['is_gold'],
        'PatientSex': m['PatientSex'],
        'n_series_on_disk': m['n_series_on_disk'],
        'n_series_stored': len(m['series']),
        'n_slices_stored': sum(s['n_slices_stored'] for s in m['series'].values()),
        'n_decode_failures': len(m['decode_failures']),
    }
    for m in meta_rows
])
manifest.to_csv(f'/kaggle/working/prep_manifest_shard{SHARD_INDEX}.csv', index=False)
series_meta_df.to_csv(f'/kaggle/working/prep_series_meta_shard{SHARD_INDEX}.csv', index=False)
# explicit columns: an empty DataFrame writes a headerless file that read_csv
# then rejects, and the zero-failure case is the one we expect most
pd.DataFrame(failed_studies, columns=['StudyInstanceUID', 'error']).to_csv(
    f'/kaggle/working/prep_failed_shard{SHARD_INDEX}.csv', index=False)

if SHARD_INDEX == 0:
    pd.DataFrame({'StudyInstanceUID': sorted(gold_uids)}).to_csv(
        '/kaggle/working/gold_study_uids.csv', index=False)

print(f'{len(os.listdir(OUT_DIR))} artifacts in {OUT_DIR}; manifests written')